# M3 – Manual labeling · Khôi · 24127192
Range **2606.00734–2606.05733**. Dùng công cụ của teammate: MAKE → đối chiếu thủ công → kiểm tra → đóng gói.

Cần **5 bài, ≥20 cặp khớp/bài, ≥100 cặp khớp**. `none` là quyết định không tìm được reference; không tính vào quota cặp khớp. Không copy suggestion thành ground truth. Notebook chỉ tạo gói manual; dataset nộp đầy đủ còn cần sáu bảng và predictions theo hướng dẫn nhóm.

Chỉnh đường dẫn ở cell cấu hình. Không dùng bản M2 cũ thay cho `m2_full_v5`. Nhãn chưa được người duyệt không được gọi là manual ground truth.

## 0. Cấu hình

In [ ]:
# Đường dẫn mặc định theo HUONG_DAN_NHOM_M3.md; chỉnh theo Drive thực tế.
MSSV = "24127192"
BRANCH = "main"
RANGE_START, RANGE_END = "2606.00734", "2606.05733"
PAPERS = ["2606.02285", "2606.01753", "2606.04575", "2606.04623", "2606.04955"]
REPO_URL = "https://github.com/VincesPowder/hcmus-project-arxiv.git"
REPO_DIR = "/content/hcmus-project-arxiv"
BIBTEX_DIR = "/content/drive/MyDrive/IntroDS/milestone2_outputs/m2_full_v5/exports/20261120-203957-9b2330/tables/bibtex"
REFERENCES_PARQUET = "/content/drive/MyDrive/arxiv_m2_outputs/range_2606/m1_tables/references.parquet"
SELECTION_CSV = "/content/drive/MyDrive/IntroDS/m3_selection_v3/selection.csv"
# Lưu vào thư mục cá nhân, ngoài nguồn dữ liệu dùng chung.
PERSONAL_DIR = f"/content/drive/MyDrive/M3_Khoi_{MSSV}"
OUTPUT_DIR = f"{PERSONAL_DIR}/sheets"
FILLED_DIR = f"{PERSONAL_DIR}/filled"
LABELS_OUT = f"{PERSONAL_DIR}/labels_manual_{MSSV}.csv"
PACKAGE_DIR = f"{PERSONAL_DIR}/hf_manual"
HF_REPO_ID = ""  # username/dataset-name; điền trước bước upload
print(MSSV, RANGE_START, RANGE_END, PAPERS, sep="\n")

## 1. Kết nối Drive, chuẩn bị repo và kiểm tra môi trường

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
import shutil
import runpy

repo_path = Path(REPO_DIR)

if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
        check=True
    )
else:
    print("Repo đã tồn tại; giữ nguyên working tree, KHÔNG pull/reset để không ghi đè file manual_match.py.")
    current_branch = subprocess.check_output(
        ["git", "-C", REPO_DIR, "branch", "--show-current"],
        text=True
    ).strip()
    if current_branch != BRANCH:
        raise RuntimeError(
            f"Repo đang ở branch {current_branch!r}, cần {BRANCH!r}. "
            "Không tự checkout để tránh làm mất thay đổi local."
        )

os.chdir(REPO_DIR)

LABEL_SCRIPT = "src/labeling/label_sheet.py"
MATCH_SCRIPT = "src/labeling/manual_match.py"
SRC_DIR = str(Path(REPO_DIR) / "src")

# Script con cần nhìn thấy src/text_clean.py.
os.environ["PYTHONPATH"] = SRC_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Cài dependencies tối thiểu cho notebook/data.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pandas", "pyarrow"],
    check=True
)

for required_file in [LABEL_SCRIPT, MATCH_SCRIPT, "src/text_clean.py"]:
    if not Path(required_file).is_file():
        raise FileNotFoundError(
            f"Thiếu {required_file}. Repo/branch hiện tại chưa có đủ file."
        )
    print("FOUND:", required_file)

for required_path in [BIBTEX_DIR, REFERENCES_PARQUET, SELECTION_CSV]:
    if not Path(required_path).exists():
        raise FileNotFoundError(f"Không tìm thấy dữ liệu đầu vào: {required_path}")
    print("FOUND:", required_path)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(FILLED_DIR).mkdir(parents=True, exist_ok=True)
Path(REPO_DIR, "labels").mkdir(parents=True, exist_ok=True)

# Compile các file; lỗi cú pháp sẽ được báo ngay tại đây.
subprocess.run(
    [sys.executable, "-m", "py_compile", LABEL_SCRIPT, MATCH_SCRIPT, "src/text_clean.py"],
    check=True
)

print("\nENVIRONMENT CHECK PASSED")
print("Repo:", REPO_DIR)
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

# Selection là nguồn quyết định; không chạy lại split.py.
selection = pd.read_csv(SELECTION_CSV, dtype=str, keep_default_na=False)
mine = selection[(selection.owner == MSSV) & (selection.kind == "manual")]
if len(mine) != 5 or set(mine.arxiv_id) != set(PAPERS):
    raise ValueError("Selection hiện tại khác 5 bài đã chốt; đối chiếu selection v3 trước khi tiếp tục.")
if not mine.arxiv_id.between(RANGE_START, RANGE_END).all():
    raise ValueError("Selection có bài ngoài range của Khôi")

## 1.1. Kiểm tra CLI của `label_sheet.py`

Cell này chỉ đọc trợ giúp CLI, không tạo hay sửa CSV. Nếu lệnh help báo lỗi, dừng và gửi output.

In [ ]:
help_result = subprocess.run(
    [sys.executable, LABEL_SCRIPT, "--help"],
    cwd=REPO_DIR,
    env=os.environ.copy(),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print(help_result.stdout)
if help_result.returncode != 0:
    raise RuntimeError("Không đọc được CLI help của label_sheet.py; chưa chạy MAKE.")


## 2. MAKE – tạo citation sheets và references sheets

Chỉ chạy cell này sau khi bước kiểm tra môi trường thành công. Output được in trực tiếp để nếu lỗi sẽ thấy traceback gốc.

In [ ]:
make_cmd = [
    sys.executable, LABEL_SCRIPT, "make",
    "--mssv", MSSV,
    "--bibtex", BIBTEX_DIR,
    "--references", REFERENCES_PARQUET,
    "--selection", SELECTION_CSV,
    "--out-dir", OUTPUT_DIR,
    "--papers", *PAPERS,
]

print("MAKE command:")
print(" ".join(make_cmd))
make_result = subprocess.run(
    make_cmd,
    cwd=REPO_DIR,
    env=os.environ.copy(),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print(make_result.stdout)

if make_result.returncode != 0:
    raise RuntimeError(
        f"MAKE thất bại (exit={make_result.returncode}). "
        "Đọc traceback ngay phía trên; không chạy các bước sau."
    )

print("MAKE hoàn tất.")


## 2.1. Kiểm tra CSV do MAKE tạo

In [ ]:
for paper in PAPERS:
    citation_file = Path(OUTPUT_DIR) / f"{paper}.csv"
    refs_file = Path(OUTPUT_DIR) / f"{paper}_references.csv"

    if not citation_file.is_file():
        raise FileNotFoundError(f"MAKE chưa tạo citation CSV: {citation_file}")
    if not refs_file.is_file():
        raise FileNotFoundError(f"MAKE chưa tạo references CSV: {refs_file}")

    citations = pd.read_csv(citation_file, dtype=str, keep_default_na=False)
    refs = pd.read_csv(refs_file, dtype=str, keep_default_na=False)

    print(f"\n{paper}")
    print("  citations:", len(citations), "| columns:", citations.columns.tolist())
    print("  references:", len(refs), "| columns:", refs.columns.tolist())

    required_citation_cols = {
        "citation_key", "entry_title", "entry_authors",
        "entry_year", "best_suggestion", "ref_arxiv_id",
        "match_detail", "note"
    }
    missing = required_citation_cols - set(citations.columns)
    if missing:
        raise ValueError(f"{paper}: citation CSV thiếu cột {sorted(missing)}")

    required_ref_cols = {
        "ref_arxiv_id", "ref_title", "ref_authors", "ref_submission_date"
    }
    missing_ref = required_ref_cols - set(refs.columns)
    if missing_ref:
        raise ValueError(f"{paper}: references CSV thiếu cột {sorted(missing_ref)}")

print("\nMAKE CSV validation passed.")


## 2.2. Chuẩn bị thư mục filled

Sao chép citation CSV mới sang thư mục filled **chỉ khi file filled chưa tồn tại**. Nếu bạn đã bắt đầu gán nhãn, file cũ được giữ nguyên.

In [ ]:
for paper in PAPERS:
    source = Path(OUTPUT_DIR) / f"{paper}.csv"
    target = Path(FILLED_DIR) / f"{paper}.csv"

    if target.exists():
        print("KEEP existing filled file:", target)
    else:
        shutil.copy2(source, target)
        print("CREATED filled file:", target)

print("\nFilled directory:", FILLED_DIR)


## 3. MANUAL MATCHING

Chạy cell này để gán nhãn lần lượt 5 paper. Mỗi paper dùng:
- citation CSV trong `FILLED_DIR`;
- references CSV trong `OUTPUT_DIR`.

Trong từng paper:
- `1`–`5`: chọn candidate;
- `m`: nhập ID thủ công;
- `n`: xác nhận không có reference;
- `s`: bỏ qua citation;
- `q`: kết thúc paper hiện tại và lưu CSV.

**Lưu ý:** Nếu bạn dừng hẳn phiên Colab giữa lúc đang gán một paper, hãy nhập `q` trước để lưu các thay đổi của phiên đó.

In [ ]:
import sys
import runpy
from pathlib import Path

for paper in PAPERS:
    citation_csv = str(Path(FILLED_DIR) / f"{paper}.csv")
    references_csv = str(Path(OUTPUT_DIR) / f"{paper}_references.csv")

    if not Path(citation_csv).is_file():
        raise FileNotFoundError(citation_csv)
    if not Path(references_csv).is_file():
        raise FileNotFoundError(references_csv)

    print("\n" + "#" * 100)
    print("BẮT ĐẦU MANUAL MATCHING:", paper)
    print("#" * 100)

    old_argv = sys.argv[:]
    try:
        sys.argv = [
            MATCH_SCRIPT,
            "--csv", citation_csv,
            "--references", references_csv,
        ]
        runpy.run_path(MATCH_SCRIPT, run_name="__main__")
    finally:
        sys.argv = old_argv

    print("\nĐã kết thúc phiên matching:", paper)
    print("CSV hiện tại:", citation_csv)

print("\nĐã chạy hết danh sách paper.")

## 3.1. Kiểm tra tiến độ gán nhãn

In [ ]:
sheet_api = runpy.run_path(str(Path(REPO_DIR) / LABEL_SCRIPT))
fix_id = sheet_api["fix_id"]
progress = []
for paper in PAPERS:
    path = Path(FILLED_DIR) / f"{paper}.csv"
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    values = df.ref_arxiv_id.map(fix_id)
    matched = int((~values.isin(["", "none"])).sum())
    progress.append((paper, len(df), matched, int(values.eq("none").sum()), int(values.eq("").sum())))
progress_df = pd.DataFrame(progress, columns=["arxiv_id", "entries", "matched", "none", "pending"])
display(progress_df)
print("Cặp khớp:", int(progress_df.matched.sum()), "/ tối thiểu 100")

## 4. Kiểm tra quota và đối chiếu nguồn
Chỉ cặp khớp được tính quota. Kiểm tra key thật trong BibTeX, ID bài, reference cùng bài, trùng key và bằng chứng quyết định. Chưa đủ điều kiện thì dừng, giữ nguyên sheets để tiếp tục.

In [ ]:
def validate_manual():
    bib = sheet_api["read_rows"](BIBTEX_DIR, PAPERS)
    refs = sheet_api["read_rows"](REFERENCES_PARQUET, PAPERS)
    problems, total = [], 0
    for paper in PAPERS:
        df = pd.read_csv(Path(FILLED_DIR) / f"{paper}.csv", dtype=str, keep_default_na=False)
        if not df.arxiv_id.map(fix_id).eq(paper).all():
            problems.append(f"{paper}: arxiv_id trong sheet sai")
        if df.citation_key.str.strip().eq("").any() or df.citation_key.duplicated().any():
            problems.append(f"{paper}: citation_key rỗng/trùng")
        keys = set(bib.loc[bib.arxiv_id.eq(paper), "citation_key"])
        if not set(df.citation_key).issubset(keys):
            problems.append(f"{paper}: key không có trong BibTeX nguồn")
        valid_refs = set(refs.loc[refs.arxiv_id.eq(paper), "ref_arxiv_id"].map(fix_id)) - {"", "none"}
        values = df.ref_arxiv_id.map(fix_id)
        positive = ~values.isin(["", "none"])
        if not set(values[positive]).issubset(valid_refs):
            problems.append(f"{paper}: reference không thuộc cùng bài")
        decided = values.ne("")
        if df.loc[decided, "match_detail"].str.strip().eq("").any():
            problems.append(f"{paper}: quyết định thiếu match_detail")
        count = int(positive.sum())
        total += count
        print(paper, count, "cặp khớp")
        if count < 20:
            problems.append(f"{paper}: {count}/20 cặp khớp")
    if total < 100:
        problems.append(f"Tổng {total}/100 cặp khớp")
    if problems:
        raise ValueError("\n".join(problems))
    print("PASS: đủ quota và hợp lệ theo nguồn; nội dung quyết định vẫn cần người duyệt.")

validate_manual()


## 4.1. Collect nhãn thành file tổng

In [ ]:
collect_cmd = [
    sys.executable, LABEL_SCRIPT, "collect",
    "--mssv", MSSV,
    "--sheets-dir", FILLED_DIR,
    "--references", REFERENCES_PARQUET,
    "--selection", SELECTION_CSV,
    "--out", LABELS_OUT,
]

print("COLLECT command:")
print(" ".join(collect_cmd))

collect_result = subprocess.run(
    collect_cmd,
    cwd=REPO_DIR,
    env=os.environ.copy(),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
print(collect_result.stdout)

if collect_result.returncode != 0:
    raise RuntimeError(
        f"COLLECT thất bại (exit={collect_result.returncode}). "
        "Đọc traceback ngay phía trên."
    )

print("COLLECT hoàn tất:", LABELS_OUT)


## 4.2. Kiểm tra file labels cuối cùng

In [ ]:
labels_path = Path(LABELS_OUT)

if not labels_path.exists():
    raise FileNotFoundError(f"Không thấy file output: {labels_path}")

labels = pd.read_csv(
    labels_path,
    dtype=str,
    keep_default_na=False
)

print("Output:", labels_path)
print("Rows:", len(labels))
print("Columns:", labels.columns.tolist())
display(labels.head(10))

# Schema đúng của MANUAL LABELS
expected = [
    "arxiv_id",
    "citation_key",
    "ref_arxiv_id",
    "source",
    "annotator"
]

missing = [c for c in expected if c not in labels.columns]

if missing:
    raise ValueError(
        f"File manual labels thiếu các cột: {missing}"
    )

# Kiểm tra tối thiểu của M3
matched_labels = labels[labels.ref_arxiv_id.str.strip().ne("")]
counts = matched_labels.groupby("arxiv_id").size()

if len(counts) < 5:
    raise ValueError(
        f"Chỉ có {len(counts)} publication, cần ít nhất 5."
    )

if (counts < 20).any():
    raise ValueError(
        f"Có publication dưới 20 citation:\n{counts[counts < 20]}"
    )

if len(matched_labels) < 100:
    raise ValueError(
        f"Tổng chỉ có {len(labels)} labels, cần ít nhất 100."
    )

print("\nManual labels schema check passed.")
print(f"Total labels: {len(labels)}")

print("\nLabels per publication:")
print(counts)

## 5. Đóng gói manual cho Hugging Face
Xuất riêng cặp matching theo schema 4 cột của module `m3`; giữ file teammate 5 cột và sheets trong Drive để audit. Các dòng `none` chỉ nằm trong bản review, không đưa groundtruth rỗng vào labels matching. Gói này là **phần manual**, chưa phải sáu bảng dataset nộp cuối.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
validate_manual()
subprocess.run(collect_cmd, cwd=REPO_DIR, env=os.environ.copy(), check=True)
labels = pd.read_csv(LABELS_OUT, dtype=str, keep_default_na=False)
matched_labels = labels[labels.ref_arxiv_id.str.strip().ne("")]
counts = matched_labels.groupby("arxiv_id").size()
package = Path(PACKAGE_DIR)
package.mkdir(parents=True, exist_ok=True)
for paper in PAPERS:
    if paper not in set(matched_labels.arxiv_id):
        raise ValueError(f"Thiếu {paper}")
if set(matched_labels.arxiv_id) != set(PAPERS) or (counts < 20).any() or len(matched_labels) < 100:
    raise ValueError("Chưa đủ quota")
matching = matched_labels.rename(columns={"ref_arxiv_id": "groundtruth_arxiv_id", "source": "label_source"})[
    ["arxiv_id", "citation_key", "groundtruth_arxiv_id", "label_source"]
]
partitions = mine.set_index("arxiv_id")["partition"]
for partition in ["train", "valid", "test"]:
    subset = matching[matching.arxiv_id.map(partitions).eq(partition)]
    if subset.empty:
        raise ValueError(f"Thiếu manual partition {partition}")
    subset.to_parquet(package / f"manual_{partition}.parquet", index=False)
matching.to_csv(package / "labels_manual.csv", index=False)
manifest = {
    "status": "manual_component_only", "owner": MSSV,
    "range": [RANGE_START, RANGE_END], "papers": PAPERS,
    "matched_pairs": len(matching), "counts": counts.to_dict(),
    "code_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "selection_sha256": hashlib.sha256(Path(SELECTION_CSV).read_bytes()).hexdigest(),
    "reviewed_labels_sha256": hashlib.sha256(Path(LABELS_OUT).read_bytes()).hexdigest(),
    "bibtex_source": BIBTEX_DIR, "references_source": REFERENCES_PARQUET,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "limitation": "Quota/schema checks do not establish correctness of human decisions. Full submission also requires M1/M2 tables and model predictions."
}
(package / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
card = f"""---
language:
- en
task_categories:
- text-retrieval
configs:
- config_name: manual
  data_files:
  - split: train
    path: manual_train.parquet
  - split: validation
    path: manual_valid.parquet
  - split: test
    path: manual_test.parquet
---
# arXiv M3 manual matching – {MSSV}

Manual annotation component for range `{RANGE_START}`–`{RANGE_END}`.
Contains {len(matching)} positive matching pairs across five publications.
Train/validation/test preserve the frozen selection.csv assignments; no re-splitting is performed.

Columns: arxiv_id, citation_key, groundtruth_arxiv_id, label_source.
Unreviewed entries and explicit no-match decisions are excluded from this positive-pair table.
See manifest.json for code and selection provenance. This package alone is not the complete six-table course submission.
"""
(package / "README.md").write_text(card, encoding="utf-8")
print("Prepared:", package)

## 6. Upload phần manual (chỉ chạy sau khi đã duyệt)
Điền `HF_REPO_ID`. Đăng nhập qua hộp thoại Hugging Face; không ghi token vào notebook hoặc chat. Bước này chỉ upload các file được liệt kê. Chưa có `predictions` và sáu bảng thì chưa coi là hoàn tất nộp bài.

In [ ]:
if not HF_REPO_ID or HF_REPO_ID.count("/") != 1:
    raise ValueError("Điền HF_REPO_ID dạng username/dataset-name ở cell cấu hình")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
from huggingface_hub import HfApi, notebook_login
notebook_login()
api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
result = api.upload_folder(
    repo_id=HF_REPO_ID, repo_type="dataset", folder_path=PACKAGE_DIR,
    allow_patterns=["manual_*.parquet", "labels_manual.csv", "manifest.json", "README.md"],
    commit_message=f"Add reviewed M3 manual matching pairs for {MSSV}",
)
print(result)